# 8.2 msSanitizer 与 msDebug

## 小节概述

本节所有命令都从 `01_debugging_tools/` 目录执行。故障程序可能返回非零；Notebook 会先捕获真实退出码，再检查 MODE 对应的诊断文本。只有诊断与故障匹配时才写 `EXPECTED_DIAGNOSTIC`，不会用 `set -e` 把预期故障误判为 Notebook 失败。


## 1. 故障代码

<table style="text-align:left; margin-left:0;">
<tr><th>MODE</th><th>故障设计</th><th>应观察到的诊断</th></tr>
<tr><td>memcheck</td><td>把 2048 个 float 的 DataCopy 改成 4096 个</td><td>illegal write</td></tr>
<tr><td>racecheck</td><td>删除 MTE2→Vector 的同步</td><td>Potential RAW hazard</td></tr>
<tr><td>initcheck</td><td>未初始化 zLocal 就搬到 GM</td><td>uninitialized read</td></tr>
<tr><td>synccheck</td><td>增加无 WaitFlag 的 SetFlag</td><td>Unpaired set_flag</td></tr>
</table>


下面是四个故障分支的最小语义片段；完整源码位于 `src/add_sanitizer.asc`。

```cpp
// memcheck：搬运长度超过 LocalTensor
AscendC::DataCopy(yLocal, xGm, 4096);

// racecheck：缺少 MTE2 -> Vector 的 WaitFlag
AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(eventId);

// initcheck：zLocal 尚未初始化就写回 GM
AscendC::DataCopy(zGm, zLocal, 2048);

// synccheck：SetFlag 没有对应 WaitFlag
AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(eventId);
```

实际分支由 `LAB05_FAULT_MODE` 编译定义选择；判断结果必须回到当前构建目录对应的源码行和诊断日志。


## 2. 独立 ASC 可执行程序

普通 CMake 目标直接把插桩选项加到目标：

```cmake
target_compile_options(lab05_add_sanitizer PRIVATE --sanitizer)
```

下面依次构建并运行四个 MODE。每个日志都保存在对应构建目录；工具退出码与诊断签名分别记录。最后再运行未插桩 baseline，确认正常路径为 `PASS`。


In [ ]:
%%bash
set -euo pipefail

signature_for() {
  case "$1" in
    memcheck)  echo 'illegal (read|write)|out[- ]of[- ]bounds' ;;
    racecheck) echo 'RAW hazard|data race' ;;
    initcheck) echo 'uninitialized' ;;
    synccheck) echo 'unpaired[ _-]*set_flag|unpaired set_flag' ;;
    *) return 2 ;;
  esac
}

run_expected() {
  local mode="$1" binary="$2" log="$3" pattern rc
  pattern="$(signature_for "$mode")"
  set +e
  mssanitizer --tool="$mode" "$binary" 2>&1 | tee "$log"
  rc="${PIPESTATUS[0]}"
  set -e
  if ! grep -Eiq "$pattern" "$log"; then
    echo "SANITIZER_RESULT status=FAIL mode=$mode exit_code=$rc reason=missing_signature" >&2
    return 1
  fi
  echo "SANITIZER_RESULT status=EXPECTED_DIAGNOSTIC mode=$mode exit_code=$rc log=$log"
}

for MODE in memcheck racecheck initcheck synccheck; do
  BUILD="build/standalone-$MODE"
  cmake -S src -B "$BUILD" \
    -DLAB05_FAULT_MODE="$MODE" -DLAB05_ENABLE_SANITIZER=ON
  cmake --build "$BUILD" -j
  run_expected "$MODE" "$BUILD/lab05_add_sanitizer" "$BUILD/mssanitizer.log"
done

BASELINE="build/standalone-baseline"
cmake -S src -B "$BASELINE" \
  -DLAB05_FAULT_MODE=baseline -DLAB05_ENABLE_SANITIZER=OFF
cmake --build "$BASELINE" -j
"$BASELINE/lab05_add_sanitizer"
echo "BASELINE_STATUS=PASS"


每个故障的完成信号是 `status=EXPECTED_DIAGNOSTIC`，并同时保留实际 `exit_code` 和日志路径；不能只写“程序报错”。baseline 必须正常返回并输出 `BASELINE_STATUS=PASS`。若故障日志没有匹配当前 MODE 的核心签名，Cell 必须失败。


## 3. 标准算子工程

标准工程的 Kernel 目标由 CANN 构建系统创建，因此采用当前版本提供的辅助函数：

```cmake
if(COMMAND add_ops_compile_options)
  add_ops_compile_options(ALL OPTIONS -sanitizer)
elseif(COMMAND npu_op_kernel_options)
  npu_op_kernel_options(ascendc_kernels ALL OPTIONS -sanitizer)
endif()
```

两者都把 `-sanitizer` 送入 Kernel 编译；区别只是 CANN 版本提供的函数名不同。Host 源码或普通 C++ 目标不应冒充 Kernel 插桩位置。


标准工程把 `LAB05_OPERATOR_FAULT_MODE` 转成 Kernel 编译定义，并在开启实验插桩时调用上述辅助函数。完整配置位于 `src/operator_project_sanitizer/op_kernel/CMakeLists.txt`；无需在 Notebook 中展开整份文件。


下面为四个 MODE 分别创建构建目录和隔离安装目录，运行 ACLNN Host，并用同一诊断签名判断 `EXPECTED_DIAGNOSTIC`。隔离安装目录来自 `mktemp`，退出 Cell 时才清理；课程不会修改系统 OPP。


In [ ]:
%%bash
set -euo pipefail
ROOT="$PWD/src/operator_project_sanitizer"
INSTALL_ROOT="$(mktemp -d "${TMPDIR:-/tmp}/cann-debug-operator.XXXXXX")"
trap 'chmod -R u+rwX -- "$INSTALL_ROOT" 2>/dev/null || true; rm -rf -- "$INSTALL_ROOT"' EXIT

signature_for() {
  case "$1" in
    memcheck)  echo 'illegal (read|write)|out[- ]of[- ]bounds' ;;
    racecheck) echo 'RAW hazard|data race' ;;
    initcheck) echo 'uninitialized' ;;
    synccheck) echo 'unpaired[ _-]*set_flag|unpaired set_flag' ;;
    *) return 2 ;;
  esac
}

run_expected() {
  local mode="$1" binary="$2" log="$3" pattern rc
  pattern="$(signature_for "$mode")"
  set +e
  mssanitizer --tool="$mode" "$binary" 2>&1 | tee "$log"
  rc="${PIPESTATUS[0]}"
  set -e
  if ! grep -Eiq "$pattern" "$log"; then
    echo "SANITIZER_RESULT status=FAIL shape=operator mode=$mode exit_code=$rc reason=missing_signature" >&2
    return 1
  fi
  echo "SANITIZER_RESULT status=EXPECTED_DIAGNOSTIC shape=operator mode=$mode exit_code=$rc log=$log"
}

for MODE in memcheck racecheck initcheck synccheck; do
  BUILD="$ROOT/build-$MODE"
  HOST="$ROOT/test/build-$MODE"
  INSTALL="$INSTALL_ROOT/$MODE"
  mkdir -p "$INSTALL"

  cmake -S "$ROOT" -B "$BUILD" -DCMAKE_BUILD_TYPE=Release \
    -DLAB05_OPERATOR_FAULT_MODE="$MODE" -DLAB05_ENABLE_SANITIZER=ON
  cmake --build "$BUILD" --target binary package -j
  PACKAGE=("$BUILD"/custom_opp_*.run)
  [[ -f "${PACKAGE[0]}" ]] || { echo "operator package missing" >&2; exit 1; }
  "${PACKAGE[0]}" --install-path="$INSTALL"

  cmake -S "$ROOT/test" -B "$HOST" -DCMAKE_SKIP_RPATH=TRUE \
    -DLAB05_CUSTOM_OPP_ROOT="$INSTALL"
  cmake --build "$HOST" -j
  export ASCEND_CUSTOM_OPP_PATH="$INSTALL/vendors/customize"
  export LD_LIBRARY_PATH="$INSTALL/vendors/customize/op_api/lib:${LD_LIBRARY_PATH:-}"
  run_expected "$MODE" "$HOST/execute_add_msanitizer" "$BUILD/mssanitizer.log"
done


## 4. msDebug

msDebug 使用未插桩的 baseline。运行前只读检查 `/proc/debug_switch` 和 `/dev/drv_debug`；缺少工具、开关文件或设备节点读写权限时输出 `MSDEBUG_STATUS=BLOCKED` 并停止本路径，不执行任何提权或系统修改。


In [ ]:
%%bash
set -euo pipefail

BLOCKERS=()
command -v msdebug >/dev/null 2>&1 || BLOCKERS+=("msdebug_missing")
[[ -r /proc/debug_switch ]] || BLOCKERS+=("debug_switch_unreadable")
[[ -r /dev/drv_debug && -w /dev/drv_debug ]] || BLOCKERS+=("drv_debug_not_readwrite")
if (( ${#BLOCKERS[@]} > 0 )); then
  echo "MSDEBUG_STATUS=BLOCKED reasons=$(IFS=,; echo "${BLOCKERS[*]}")"
  exit 0
fi

echo "debug_switch=$(cat /proc/debug_switch)"
BUILD="build/msdebug"
cmake -S src -B "$BUILD" \
  -DLAB05_FAULT_MODE=baseline -DLAB05_ENABLE_SANITIZER=OFF
cmake --build "$BUILD" -j
LINE="$(grep -n 'AscendC::Add(' src/add_sanitizer.asc | head -n1 | cut -d: -f1)"
cat > "$BUILD/msdebug.cmd" <<EOF
breakpoint set --file add_sanitizer.asc --line $LINE
run
thread backtrace
frame info
source list
breakpoint disable 1
continue
quit
EOF

set +e
printf 'y\n' | timeout 300 msdebug --batch --source "$BUILD/msdebug.cmd" -- \
  "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/msdebug.log"
RC="${PIPESTATUS[1]}"
set -e
if [[ "$RC" -ne 0 ]]; then
  echo "MSDEBUG_STATUS=FAIL exit_code=$RC reason=command_failed log=$BUILD/msdebug.log" >&2
  exit 1
fi
grep -Eiq 'stop reason = breakpoint|breakpoint [0-9]+' "$BUILD/msdebug.log" || {
  echo "MSDEBUG_STATUS=FAIL reason=breakpoint_not_observed" >&2
  exit 1
}
grep -Eiq 'frame #[0-9]+' "$BUILD/msdebug.log" || {
  echo "MSDEBUG_STATUS=FAIL reason=backtrace_or_frame_missing" >&2
  exit 1
}
grep -Fq 'add_sanitizer.asc:' "$BUILD/msdebug.log" || {
  echo "MSDEBUG_STATUS=FAIL reason=source_location_missing" >&2
  exit 1
}
echo "MSDEBUG_STATUS=PASS line=$LINE log=$BUILD/msdebug.log"


`MSDEBUG_STATUS=PASS` 只表示本次 baseline 断点、调用栈、栈帧和源码查看均完成。前置工具或权限不足时，`BLOCKED` 必须保留阻塞原因；前置条件满足后命令执行非零则记为 `FAIL` 并保留退出码。任何情况下都不得为了得到 PASS 修改 `/proc/debug_switch` 或 `/dev/drv_debug` 权限。


## 课后实践

从 `racecheck`、`initcheck`、`synccheck` 中任选一种，在独立 ASC 与标准算子工程下各运行一次。记录问题代码、完整命令、工具退出码、核心诊断、源码位置和根因，并与 memcheck 对比。


In [ ]:
# 完成练习后按需执行；Notebook 不会自动展开答案。
!cat answer/08.02_answer.md
